# Vesuvius Challenge - Surface Detection (Inference + tuning)

This notebook targets a stronger baseline using a pretrained ensemble, sliding-window inference, TTA, and topology-aware post-processing.

**Kaggle requirements**: notebook submission, <=9h runtime, internet disabled, submission file named `submission.zip`.

## 0. Optional: offline package install (Kaggle)

In [ ]:
import os
import sys
import glob
import subprocess

whl_dir = '/kaggle/input/vsdetection-packages-offline-installer-only/whls'
if os.path.isdir(whl_dir):
    patterns = [
        'keras_nightly-*.whl',
        'tifffile-*.whl',
        'imagecodecs-*.whl',
        'medicai-*.whl',
    ]
    wheels = []
    for p in patterns:
        wheels.extend(glob.glob(os.path.join(whl_dir, p)))
    if wheels:
        cmd = [sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', whl_dir, *wheels]
        subprocess.check_call(cmd)
    else:
        print('No wheels found in', whl_dir)
else:
    print('Wheel directory not found:', whl_dir)

## 1. Imports and config

In [ ]:
import warnings
from pathlib import Path

os.environ['KERAS_BACKEND'] = 'jax'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import zipfile
import tifffile
import scipy.ndimage as ndi
from skimage.morphology import remove_small_objects
from matplotlib import pyplot as plt

import keras
from keras import ops
from medicai.transforms import Compose, NormalizeIntensity
from medicai.models import SegFormer, TransUNet
from medicai.utils.inference import SlidingWindowInference

SEED = 42
np.random.seed(SEED)

keras.config.backend(), keras.version()


## 2. Paths and metadata

In [ ]:
DATA_DIR = Path('/kaggle/input/vesuvius-challenge-surface-detection')
TRAIN_CSV = DATA_DIR / 'train.csv'
TEST_CSV = DATA_DIR / 'test.csv'
TRAIN_IMG_DIR = DATA_DIR / 'train_images'
TRAIN_LABEL_DIR = DATA_DIR / 'train_labels'
TEST_IMG_DIR = DATA_DIR / 'test_images'

OUTPUT_DIR = Path('/kaggle/working')
SUBMISSION_ZIP = OUTPUT_DIR / 'submission.zip'

MODEL_DIR_CANDIDATES = [
    '/kaggle/input/vsd-model/keras',
    '/kaggle/input/vsd-model',
]

def first_existing(paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return None

model_dir = first_existing(MODEL_DIR_CANDIDATES)
print('Model dir:', model_dir)

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
display(train_df.head())
display(test_df.head())

## 3. Preprocessing helpers

In [ ]:
def val_transformation(image):
    # Robust percentile normalization per volume
    img = image.astype(np.float32)
    p1, p99 = np.percentile(img, (1, 99))
    if p99 > p1:
        img = (img - p1) / (p99 - p1)
    img = np.clip(img, 0.0, 1.0)
    return img

def load_volume(path):
    vol = tifffile.imread(str(path))
    vol = vol.astype(np.float32)
    vol = vol[None, ..., None]  # (1, D, H, W, 1)
    return vol

def load_label(path):
    return tifffile.imread(str(path))


## 4. Model ensemble

In [ ]:
USE_TTA = True
TTA_FLIP_AXES = [1, 2, 3]  # D/H/W for (1, D, H, W, 1)
TTA_ROT_K = [1, 2, 3]      # 90-degree rotations in (H, W)
FOREGROUND_INDEX = 1

POSTPROCESS = dict(
    T_low=0.50,
    T_high=0.90,
    z_radius=1,
    xy_radius=0,
    dust_min_size=100,
)

DEFAULT_SEGFORMER_SHAPE = (128, 128, 128)
DEFAULT_TRANSUNET_SHAPE = (160, 160, 160)

MODEL_SPECS = [
    dict(
        name='default_v1',
        kind='transunet',
        encoder='seresnext50',
        weight_candidates=[
            'default/1/*160*.h5',
            'default/1/*128*.h5',
            'default/1/*.h5',
        ],
        default_input_shape=DEFAULT_TRANSUNET_SHAPE,
        default_num_classes=2,
        weight=0.3,
    ),
    dict(
        name='default_v2',
        kind='transunet',
        encoder='seresnext50',
        weight_candidates=[
            'default/2/*160*.h5',
            'default/2/*128*.h5',
            'default/2/*.h5',
        ],
        default_input_shape=DEFAULT_TRANSUNET_SHAPE,
        default_num_classes=2,
        weight=0.4,
    ),
    dict(
        name='segformer_mit_b2_v1',
        kind='segformer',
        encoder='mit_b2',
        weight_candidates=[
            'segformer.mit.b2/1/*.h5',
        ],
        default_input_shape=DEFAULT_SEGFORMER_SHAPE,
        default_num_classes=2,
        weight=0.4,
    ),
    dict(
        name='segformer_mit_b2_v2',
        kind='segformer',
        encoder='mit_b2',
        weight_candidates=[
            'segformer.mit.b2/2/*.h5',
        ],
        default_input_shape=DEFAULT_SEGFORMER_SHAPE,
        default_num_classes=2,
        weight=0.5,
    ),
    dict(
        name='segformer_mit_b4_v1',
        kind='segformer',
        encoder='mit_b4',
        weight_candidates=[
            'segformer.mit.b4/1/*.h5',
        ],
        default_input_shape=DEFAULT_SEGFORMER_SHAPE,
        default_num_classes=2,
        weight=0.5,
    ),
    dict(
        name='transunetseresnext_v2',
        kind='transunet',
        encoder='seresnext50',
        weight_candidates=[
            'transunetseresnext/2/*160*.h5',
            'transunetseresnext/2/*128*.h5',
            'transunetseresnext/2/*.h5',
        ],
        default_input_shape=DEFAULT_TRANSUNET_SHAPE,
        default_num_classes=2,
        weight=0.6,
    ),
    dict(
        name='transunet_v2',
        kind='transunet',
        encoder='seresnext50',
        weight_candidates=[
            'transunet/2/*160*.h5',
            'transunet/2/*128*.h5',
            'transunet/2/*.h5',
        ],
        default_input_shape=DEFAULT_TRANSUNET_SHAPE,
        default_num_classes=2,
        weight=0.8,
    ),
    dict(
        name='transunet_v3',
        kind='transunet',
        encoder='seresnext50',
        weight_candidates=[
            'transunet/3/*160*.h5',
            'transunet/3/*.h5',
        ],
        default_input_shape=DEFAULT_TRANSUNET_SHAPE,
        default_num_classes=3,
        weight=1.0,
    ),
]

def resolve_weights(model_dir, patterns):
    for pattern in patterns:
        matches = sorted(model_dir.glob(pattern))
        if matches:
            return matches[0]
    return None

def infer_input_shape(weights_path, default_shape):
    name = weights_path.name
    if '160' in name:
        return (160, 160, 160)
    if '128' in name:
        return (128, 128, 128)
    return default_shape

def infer_num_classes(weights_path, default_classes):
    name = weights_path.name
    if '160' in name:
        return 3
    if '128' in name:
        return 2
    return default_classes

def build_model(kind, encoder, input_shape, num_classes):
    input_shape = tuple(input_shape) + (1,)
    if kind == 'transunet':
        model = TransUNet(
            input_shape=input_shape,
            encoder_name=encoder,
            classifier_activation=None,
            num_classes=num_classes,
        )
    elif kind == 'segformer':
        model = SegFormer(
            input_shape=input_shape,
            encoder_name=encoder,
            classifier_activation=None,
            num_classes=num_classes,
        )
    else:
        raise ValueError(f'Unknown model kind: {kind}')
    return model

def load_models(model_specs):
    if model_dir is None:
        raise RuntimeError('Model directory not found. Add the vsd-model dataset as an input.')

    loaded = []
    for spec in model_specs:
        weights_path = resolve_weights(model_dir, spec['weight_candidates'])
        if weights_path is None:
            print('[skip] Missing weights for', spec['name'])
            continue

        input_shape = spec.get('input_shape') or infer_input_shape(weights_path, spec['default_input_shape'])
        num_classes = spec.get('num_classes') or infer_num_classes(weights_path, spec['default_num_classes'])

        model = build_model(spec['kind'], spec['encoder'], input_shape, num_classes)
        model.load_weights(str(weights_path))

        swi = SlidingWindowInference(
            model,
            num_classes=num_classes,
            roi_size=input_shape,
            sw_batch_size=1,
            mode='gaussian',
            overlap=0.5,
        )

        spec = dict(spec)
        spec['weights_path'] = str(weights_path)
        spec['input_shape'] = input_shape
        spec['num_classes'] = num_classes

        loaded.append((spec, model, swi))
        print('[load]', spec['name'], '->', weights_path.name, 'shape', input_shape, 'classes', num_classes)

    if not loaded:
        raise RuntimeError('No models loaded. Check MODEL_DIR and MODEL_SPECS.')

    return loaded

models = load_models(MODEL_SPECS)
print('Loaded models:', [m[0]['name'] for m in models])

## 5. Inference helpers (TTA + postprocess)

In [ ]:
def _prob_from_logits(logits, foreground_index=1):
    if logits.shape[-1] == 1:
        prob = ops.sigmoid(logits)[..., 0]
    else:
        prob = ops.softmax(logits, axis=-1)[..., foreground_index]
    return ops.convert_to_numpy(prob)

def predict_with_tta(inputs, swi, foreground_index=1, use_tta=True):
    probs = []
    logits = swi(inputs)
    probs.append(_prob_from_logits(logits, foreground_index))

    if use_tta:
        for axis in TTA_FLIP_AXES:
            img_f = np.flip(inputs, axis=axis)
            p = _prob_from_logits(swi(img_f), foreground_index)
            p = np.flip(p, axis=axis)
            probs.append(p)
        for k in TTA_ROT_K:
            img_r = np.rot90(inputs, k=k, axes=(2, 3))
            p = _prob_from_logits(swi(img_r), foreground_index)
            p = np.rot90(p, k=-k, axes=(2, 3))
            probs.append(p)

    mean_prob = np.mean(probs, axis=0)
    return mean_prob.squeeze()

def ensemble_predict(volume, use_tta=True):
    total = 0.0
    wsum = 0.0
    for spec, model, swi in models:
        prob = predict_with_tta(
            volume,
            swi,
            foreground_index=FOREGROUND_INDEX,
            use_tta=use_tta,
        )
        total += prob * spec['weight']
        wsum += spec['weight']
    return total / max(wsum, 1e-6)

def build_anisotropic_struct(z_radius: int, xy_radius: int):
    z, r = z_radius, xy_radius
    if z == 0 and r == 0:
        return None
    if z == 0 and r > 0:
        size = 2 * r + 1
        struct = np.zeros((1, size, size), dtype=bool)
        cy, cx = r, r
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[0, cy + dy, cx + dx] = True
        return struct
    if z > 0 and r == 0:
        struct = np.zeros((2 * z + 1, 1, 1), dtype=bool)
        struct[:, 0, 0] = True
        return struct
    depth = 2 * z + 1
    size = 2 * r + 1
    struct = np.zeros((depth, size, size), dtype=bool)
    cz, cy, cx = z, r, r
    for dz in range(-z, z + 1):
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[cz + dz, cy + dy, cx + dx] = True
    return struct

def topo_postprocess(
    prob,
    T_low=0.50,
    T_high=0.90,
    z_radius=1,
    xy_radius=0,
    dust_min_size=100,
):
    strong = prob >= T_high
    weak = prob >= T_low

    if not strong.any():
        return np.zeros_like(prob, dtype=np.uint8)

    struct_hyst = ndi.generate_binary_structure(3, 3)
    mask = ndi.binary_propagation(strong, mask=weak, structure=struct_hyst)

    if not mask.any():
        return np.zeros_like(prob, dtype=np.uint8)

    if z_radius > 0 or xy_radius > 0:
        struct_close = build_anisotropic_struct(z_radius, xy_radius)
        if struct_close is not None:
            mask = ndi.binary_closing(mask, structure=struct_close)

    if dust_min_size > 0:
        mask = remove_small_objects(mask.astype(bool), min_size=dust_min_size)

    return mask.astype(np.uint8)

def inference_pipeline(volume, use_tta=True, postprocess=None):
    prob = ensemble_predict(volume, use_tta=use_tta)
    if postprocess is None:
        postprocess = POSTPROCESS
    return topo_postprocess(prob, **postprocess)


## 6. Threshold tuning (optional)
This uses a simple Dice proxy on a small train subset. It is not the official metric, but helps pick reasonable postprocess thresholds.

In [ ]:
TUNE_ENABLE = False
TUNE_NUM_SAMPLES = 2
TUNE_USE_TTA = False  # set True if you can afford extra time
TUNE_STRIDE = 2       # >1 subsamples voxels for faster scoring

param_grid = {
    'T_low': [0.40, 0.50, 0.60],
    'T_high': [0.80, 0.90],
    'z_radius': [0, 1, 2],
    'xy_radius': [0, 1],
    'dust_min_size': [0, 100, 300],
}

def dice_score(pred, target, ignore_mask=None):
    if ignore_mask is not None:
        pred = pred[~ignore_mask]
        target = target[~ignore_mask]
    pred = pred.astype(bool)
    target = target.astype(bool)
    inter = np.logical_and(pred, target).sum()
    denom = pred.sum() + target.sum()
    if denom == 0:
        return 1.0
    return 2.0 * inter / denom

def apply_stride(arr, stride):
    if stride is None or stride <= 1:
        return arr
    return arr[::stride, ::stride, ::stride]

if TUNE_ENABLE:
    sample_df = train_df.sample(TUNE_NUM_SAMPLES, random_state=SEED)
    cached = []

    for row in sample_df.itertuples(index=False):
        img_path = TRAIN_IMG_DIR / f'{row.id}.tif'
        lbl_path = TRAIN_LABEL_DIR / f'{row.id}.tif'

        volume = load_volume(img_path)
        volume = val_transformation(volume)
        prob = ensemble_predict(volume, use_tta=TUNE_USE_TTA)

        label = load_label(lbl_path)
        ignore = label == 2
        target = label == 1

        prob = apply_stride(prob, TUNE_STRIDE)
        target = apply_stride(target, TUNE_STRIDE)
        ignore = apply_stride(ignore, TUNE_STRIDE)

        cached.append((prob, target, ignore))

    best = None
    best_score = -1

    for T_low in param_grid['T_low']:
        for T_high in param_grid['T_high']:
            for z_radius in param_grid['z_radius']:
                for xy_radius in param_grid['xy_radius']:
                    for dust_min_size in param_grid['dust_min_size']:
                        scores = []
                        for prob, target, ignore in cached:
                            pred = topo_postprocess(
                                prob,
                                T_low=T_low,
                                T_high=T_high,
                                z_radius=z_radius,
                                xy_radius=xy_radius,
                                dust_min_size=dust_min_size,
                            )
                            score = dice_score(pred, target, ignore)
                            scores.append(score)
                        mean_score = float(np.mean(scores))
                        if mean_score > best_score:
                            best_score = mean_score
                            best = dict(
                                T_low=T_low,
                                T_high=T_high,
                                z_radius=z_radius,
                                xy_radius=xy_radius,
                                dust_min_size=dust_min_size,
                            )

    print('Best proxy Dice:', best_score)
    print('Best postprocess:', best)
    POSTPROCESS = best
else:
    print('Tuning disabled. Using POSTPROCESS:', POSTPROCESS)

## 7. Inference and submission

In [ ]:
tmp_dir = OUTPUT_DIR / 'preds'
tmp_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(SUBMISSION_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for row in test_df.itertuples(index=False):
        image_id = row.id
        img_path = TEST_IMG_DIR / f'{image_id}.tif'

        volume = load_volume(img_path)
        volume = val_transformation(volume)
        output = inference_pipeline(volume, use_tta=USE_TTA, postprocess=POSTPROCESS)

        out_path = tmp_dir / f'{image_id}.tif'
        tifffile.imwrite(str(out_path), output.astype(np.uint8))
        zf.write(out_path, arcname=out_path.name)
        os.remove(out_path)

print('Saved', SUBMISSION_ZIP)

## 8. Optional debug plot

In [ ]:
def plot_sample(x, y, sample_idx=0, max_slices=8):
    img = np.squeeze(x[sample_idx])
    mask = np.squeeze(y[sample_idx])
    D = img.shape[0]
    step = max(1, D // max_slices)
    slices = range(0, D, step)

    n_slices = len(slices)
    fig, axes = plt.subplots(2, n_slices, figsize=(3 * n_slices, 6))
    for i, s in enumerate(slices):
        axes[0, i].imshow(img[s], cmap='gray')
        axes[0, i].set_title(f'Slice {s}')
        axes[0, i].axis('off')

        axes[1, i].imshow(mask[s], cmap='gray')
        axes[1, i].set_title(f'Mask {s}')
        axes[1, i].axis('off')

    plt.suptitle(f'Sample {sample_idx}')
    plt.tight_layout()
    plt.show()

# Example usage (after running inference on a sample)
# plot_sample(volume, output[None], sample_idx=0, max_slices=5)